In [1]:
import optuna
import os
import sys
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

sys.path.append(os.path.abspath('..'))
from Environment.liquidation_env import LiquidationEnv

# 1. 定义目标函数 (Optuna 的核心)
def objective(trial):
    # a. 让 Optuna 在指定范围内“采样”超参数
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    n_steps = trial.suggest_categorical("n_steps", [512, 1024, 2048])
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    gamma = trial.suggest_float("gamma", 0.95, 1.0) # 测试是否需要略微的时间折扣
    
    # b. 使用这组采样的参数初始化环境和模型
    env = LiquidationEnv(lam=0.5, reward_scale=1e-3)
    model = PPO("MlpPolicy", env, 
                learning_rate=learning_rate, 
                n_steps=n_steps, 
                batch_size=batch_size, 
                gamma=gamma, 
                verbose=0)
    
    # c. 快速试跑一小段 (比如 2万步，不用完全跑完)
    try:
        model.learn(total_timesteps=20000)
    except:
        # 防止参数太离谱导致崩溃
        return -1000.0 
        
    # d. 评估这组参数的得分
    mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=5)
    
    # 清理内存
    env.close()
    del model
    
    return mean_reward

# 2. 创建并启动“学习计划” (Study)
print("开始 Optuna 超参搜索...")
# direction="maximize" 意味着我们要寻找让 reward 最大的参数
study = optuna.create_study(direction="maximize") 
# n_trials=20 表示让它尝试 20 种不同的组合 (真实论文建议设为 50-100)
study.optimize(objective, n_trials=20)

# 3. 输出打印最牛的参数
print("\n=== 搜索结束！最牛的参数组合是 ===")
print(study.best_params)
print(f"最高平均奖励: {study.best_value}")

/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-09 00:57:07.088147: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 00:57:07.195385: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775667427.220683    8422 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775667427.229645    8422 cuda_blas.cc:1418]

开始 Optuna 超参搜索...


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn


=== 搜索结束！最牛的参数组合是 ===
{'learning_rate': 0.0007696558671593552, 'n_steps': 512, 'batch_size': 256, 'gamma': 0.9632994051086053}
最高平均奖励: -0.018695952108809782
